# Colombia FAQ Chatbot - Retrieval-Augmented Generation (RAG)

**Author: Juan David Nieto**

A multilingual question-answering assistant built over a local corpus about Colombia.
It retrieves the most relevant passages with FAISS and generates grounded answers with a
free Hugging Face model. The assistant replies in the same language as the question:
**English, Portuguese, or Spanish**.

**Pipeline:** local `.txt` corpus -> multilingual embeddings -> FAISS index -> retrieval
-> language-aware generation.

> Before running: in Colab go to `Runtime` -> `Change runtime type` -> select GPU (T4),
> then `Runtime` -> `Run all`.


## 1. Check the GPU

In [1]:
!nvidia-smi

Tue Jun  2 00:48:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies

In [2]:
!pip -q install -U faiss-cpu sentence-transformers transformers accelerate langdetect ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.8 MB/s eta 0:00:00


## 3. Imports and configuration

In [3]:
import os
import json
import torch
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

CORPUS_DIR = "corpus_colombia"
HISTORY_PATH = "history.json"

EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"

TOP_K = 4
CHUNK_SIZE = 600
CHUNK_OVERLAP = 100
MAX_NEW_TOKENS = 320

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Embeddings:", EMB_MODEL)
print("Generator:", GEN_MODEL)

Device: cuda
Embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Generator: Qwen/Qwen2.5-3B-Instruct


## 4. Create the local corpus

The notebook writes the `.txt` files into `corpus_colombia/` so it is self-contained.

In [4]:
os.makedirs(CORPUS_DIR, exist_ok=True)

CORPUS = {
    "01_overview.txt": "COLOMBIA OVERVIEW\n\nColombia, officially the Republic of Colombia, is a country in northwestern South America. Its capital is Bogota. The country's name comes from the surname of Christopher Columbus and can be understood as \"the land of Columbus\".\n\nColombia borders Venezuela and Brazil to the east, Peru to the southeast, Ecuador to the southwest, and Panama to the northwest. It is the only South American country with coastlines on both the Pacific Ocean and the Caribbean Sea, which gives it a strategic position for trade and tourism.\n\nThe official language is Spanish and the currency is the Colombian peso. The population is around 52 million people. Its main cities include Bogota, Medellin, Cali, Barranquilla, and Cartagena.\n\nColombian society is diverse: it includes Indigenous, Afro-Colombian, Raizal, and mestizo communities. The country is known for its hospitality, its cheerfulness, and its resilience.\n",
    "02_geography_climate.txt": "GEOGRAPHY AND CLIMATE\n\nColombia is a tropical country, but its territory is crossed by the Andes mountains, which split into three ranges inside the country: the Western, Central, and Eastern cordilleras. Because of this, the climate changes a great deal with altitude, from warm lowlands at sea level to cold highland paramos in the mountains.\n\nSince the country lies near the equator, the length of day and night stays fairly constant throughout the year, and there are no marked seasons like in temperate zones. The lowlands have a warm, humid climate, while the mountainous areas offer mild and cold climates.\n\nThe country is organized into six natural regions: Caribbean, Pacific, Andean, Amazon, Orinoquia, and Insular. The Colombian Pacific is a very rainy and megadiverse ecosystem where the rainforest meets the sea. The Orinoquia, or Eastern Plains, has a savanna climate with one dry season and one rainy season.\n\nColombia is one of the most biodiverse countries on the planet and ranks among the top in the world for the number of bird species.\n",
    "03_history.txt": "HISTORY OF COLOMBIA\n\nBefore the Europeans arrived, the territory was home to several Indigenous peoples. Among the best known are the Muisca, in the central highlands, and the Tayrona, in the Sierra Nevada de Santa Marta, who built settlements such as Teyuna, known today as the Lost City.\n\nDuring the 16th century the Spanish conquest began, starting the colonial period. In that era, enslaved people were also brought from Africa to work in mines and plantations, leaving a deep cultural mark, especially along the coasts.\n\nThe independence process formally began on 20 July 1810 with the so-called cry of independence in Bogota. Independence was consolidated with the Battle of Boyaca, on 7 August 1819, which secured the territory's liberation from Spanish rule. After independence, Gran Colombia was formed, a union of nations led by Simon Bolivar that eventually dissolved into the present-day countries.\n",
    "04_culture_music_literature.txt": "CULTURE, MUSIC, AND LITERATURE\n\nColombia's geographic and ethnic diversity is reflected in a rich culture. Each region has its own dances, rhythms, and traditions that reaffirm the country's identity.\n\nIn literature, Gabriel Garcia Marquez stands out as a Nobel laureate and a central figure of magical realism. In his work appears Macondo, an imaginary town that became a symbol of Latin American storytelling. Colombia is also associated with the legend of El Dorado.\n\nIn music, each region contributes its own genres. In the Caribbean, the cumbia and the vallenato are emblematic. In the Eastern Plains people play the joropo, accompanied by the harp. In the Pacific, the currulao prevails, with marimba and drums of African heritage.\n\nThe country celebrates many fairs and festivals throughout the year, combining music, food, and popular traditions.\n",
    "05_food.txt": "FOOD\n\nColombian cuisine is very varied because each region adapted its dishes to local products and to Indigenous, Spanish, and African influences.\n\nFrom Antioquia comes the famous bandeja paisa, a hearty dish that usually includes beans, rice, egg, slices of fried ripe plantain, pork crackling, ground beef, blood sausage, chorizo, and avocado. In Bogota and the surrounding highlands, the ajiaco is typical, a soup of Muisca origin made with potatoes and seasoned with a herb called guascas.\n\nIf a single dish had to represent the whole country, many would mention the sancocho, a filling soup with meat, root vegetables, and plantain. In the Valle del Cauca and the Pacific, arroz atollado is prepared. On the coasts, coconut, plantain, and fish are the base of many recipes, a heritage of the Afro-descendant communities that also contributed fried foods, such as the patacon with hogao. In the Eastern Plains, ternera a la llanera, slowly roasted beef, stands out.\n",
    "06_tourism.txt": "TOURISM AND DESTINATIONS\n\nColombia offers very diverse destinations. Cartagena de Indias is the country's main tourist destination; its historic center was declared a World Heritage Site by UNESCO and preserves more than five kilometers of walls, considered the best-preserved fortification system in South America.\n\nCano Cristales, in La Macarena, is known as the river of five colors or \"the rainbow that melted\". The best time to visit is between July and November, when an endemic plant called Macarenia clavigera turns its waters red, magenta, green, and yellow.\n\nTayrona National Natural Park combines tropical rainforest and white-sand beaches. Nearby is the Lost City of the Tayrona, reached after a multi-day hike through the jungle. The Coffee Cultural Landscape, declared by UNESCO, lets visitors experience coffee culture on farms among the mountains. Other natural destinations include the Sierra Nevada del Cocuy, the Tatacoa Desert, and the Amazon region.\n\nIn 2024, Colombia received nearly seven million foreign visitors, a record figure that represented growth of about 8.5 percent over the previous year.\n",
    "07_economy.txt": "ECONOMY\n\nColombia has one of the largest economies in Latin America and a diversified model that relies on mining, agriculture, manufacturing, services, and tourism.\n\nCoffee is a national symbol: the country is one of the world's largest producers and a benchmark for exporting high-quality mild coffee, grown on the Andean slopes of the Coffee Region. Colombia is also a world leader in the production of emeralds, accounting for a very significant share of global output.\n\nThe oil sector is key to public finances, with Ecopetrol as its flagship company. Added to this are other agricultural products, mining, and a growing services sector. Tourism has become an increasingly important alternative for the country's economy.\n"
}

for name, text in CORPUS.items():
    with open(os.path.join(CORPUS_DIR, name), "w", encoding="utf-8") as f:
        f.write(text)

print("Files created:", sorted(os.listdir(CORPUS_DIR)))

Files created: ['01_overview.txt', '02_geography_climate.txt', '03_history.txt', '04_culture_music_literature.txt', '05_food.txt', '06_tourism.txt', '07_economy.txt']


## 5. Load and chunk the documents

In [5]:
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, current = [], ""
    for p in paragraphs:
        if len(current) + len(p) + 1 <= size:
            current = (current + " " + p).strip()
        else:
            if current:
                chunks.append(current)
            current = p
    if current:
        chunks.append(current)

    if overlap > 0 and len(chunks) > 1:
        overlapped = [chunks[0]]
        for i in range(1, len(chunks)):
            tail = chunks[i - 1][-overlap:]
            overlapped.append((tail + " " + chunks[i]).strip())
        chunks = overlapped
    return chunks


chunks, sources = [], []
for name in sorted(os.listdir(CORPUS_DIR)):
    with open(os.path.join(CORPUS_DIR, name), encoding="utf-8") as f:
        content = f.read()
    for ch in chunk_text(content):
        chunks.append(ch)
        sources.append(name)

print("Total chunks:", len(chunks))

Total chunks: 15


## 6. Indexing with FAISS (embeddings)

In [6]:
emb = SentenceTransformer(EMB_MODEL, device=DEVICE)

vectors = emb.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
dim = vectors.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(vectors)

print("Indexed vectors:", index.ntotal, "| dimension:", dim)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed vectors: 15 | dimension: 384


## 7. Retrieval

In [7]:
def retrieve(question, k=TOP_K):
    query = emb.encode([question], normalize_embeddings=True)
    scores, indices = index.search(query, k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        results.append({"text": chunks[idx], "source": sources[idx], "score": float(score)})
    return results


for r in retrieve("O que e Cano Cristales?"):
    print(f"[{r['source']}] ({r['score']:.2f}) {r['text'][:90]}...")

[06_tourism.txt] (0.28) , when an endemic plant called Macarenia clavigera turns its waters red, magenta, green, a...
[05_food.txt] (0.27) iaco is typical, a soup of Muisca origin made with potatoes and seasoned with a herb calle...
[04_culture_music_literature.txt] (0.26) e a symbol of Latin American storytelling. Colombia is also associated with the legend of ...
[05_food.txt] (0.20) FOOD Colombian cuisine is very varied because each region adapted its dishes to local prod...


## 8. Load the generation model

In [8]:
tok = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    model = model.to(DEVICE)

print("Model loaded:", GEN_MODEL)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-3B-Instruct


## 9. Language detection

In [9]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

LANGUAGE_NAMES = {"en": "English", "es": "Spanish", "pt": "Portuguese"}


def detect_language(text):
    try:
        code = detect(text)
    except Exception:
        code = "en"
    return LANGUAGE_NAMES.get(code, "the same language as the question")


for s in ["Where is Colombia located?", "Onde fica a Colombia?", "Donde queda Colombia?"]:
    print(f"{detect_language(s):>12}  <-  {s}")

     English  <-  Where is Colombia located?
     Spanish  <-  Onde fica a Colombia?
     Spanish  <-  Donde queda Colombia?


## 10. Generation (grounded and language-aware)

In [10]:
def build_prompt(question, context, language):
    system = (
        "You are a helpful assistant that answers questions about Colombia. "
        "Use only the information in the provided context. "
        "If the answer is not in the context, say that you do not have that information. "
        "Be clear and concise, and do not invent facts. "
        f"Write your entire answer in {language}."
    )
    user = (
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer in {language}:"
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def generate_answer(question, context, language):
    messages = build_prompt(question, context, language)
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tok.eos_token_id,
        )

    new_tokens = output[0][ids["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()

## 11. Full RAG pipeline and history

In [11]:
def save_history(record):
    history = []
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH, encoding="utf-8") as f:
            history = json.load(f)
    history.append(record)
    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)


def ask(question, k=TOP_K, save=True):
    language = detect_language(question)
    passages = retrieve(question, k)
    context = "\n\n".join(p["text"] for p in passages)
    answer = generate_answer(question, context, language)
    result = {
        "question": question,
        "answer": answer,
        "language": language,
        "sources": sorted({p["source"] for p in passages}),
    }
    if save:
        save_history(result)
    return result

## 12. Quick tests (English, Portuguese, Spanish)

In [12]:
questions = [
    "What is the capital of Colombia and which countries border it?",
    "O que leva a bandeja paisa?",
    "Que es Cano Cristales y cuando es mejor visitarlo?",
    "Quando a independencia da Colombia foi consolidada?",
    "Por que el cafe es importante para la economia colombiana?",
]

for q in questions:
    r = ask(q)
    print("Q:", r["question"])
    print(f"[{r['language']}] {r['answer']}")
    print("Sources:", ", ".join(r["sources"]))
    print("-" * 70)

Q: What is the capital of Colombia and which countries border it?
[English] The capital of Colombia is Bogotá. Colombia borders Venezuela and Brazil to the east, Peru to the southeast, Ecuador to the southwest, and Panama to the northwest.
Sources: 01_overview.txt, 02_geography_climate.txt, 03_history.txt, 07_economy.txt
----------------------------------------------------------------------
Q: O que leva a bandeja paisa?
[Portuguese] A bandeja paisa leva a bandeja, um prato grande tradicional da região de Antioquia, Colômbia. Esta refeição é conhecida por ser muito rica e variada, incluindo vários ingredientes como feijão, arroz, ovo, batatas fritas, linguiça, carne moída, chouriço, abacaxi e torradas de batata.
Sources: 04_culture_music_literature.txt, 05_food.txt, 06_tourism.txt
----------------------------------------------------------------------
Q: Que es Cano Cristales y cuando es mejor visitarlo?
[Spanish] Cano Cristales es conocido como el río de cinco colores o la araña que se

## 13. Interactive chat

In [14]:
import ipywidgets as widgets
from IPython.display import display, HTML

chat_log = widgets.Output()
text_in = widgets.Text(
    placeholder="Ask about Colombia in English, Portuguese or Spanish...",
    layout=widgets.Layout(flex="1 1 auto"),
)
send_btn = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear")


def render_message(role, text, meta=""):
    align = "flex-end" if role == "user" else "flex-start"
    bg = "#2563eb" if role == "user" else "#f1f5f9"
    color = "#ffffff" if role == "user" else "#0f172a"
    name = "You" if role == "user" else "Assistant"
    meta_html = (
        f"<div style='font-size:11px;color:#64748b;margin-top:6px'>{meta}</div>"
        if meta else ""
    )
    html = f"""
    <div style='display:flex;justify-content:{align};margin:6px 0;font-family:sans-serif'>
      <div style='max-width:80%;background:{bg};color:{color};padding:10px 14px;border-radius:14px'>
        <div style='font-size:11px;opacity:.7;margin-bottom:3px'>{name}</div>
        <div style='white-space:pre-wrap;line-height:1.45'>{text}</div>
        {meta_html}
      </div>
    </div>"""
    with chat_log:
        display(HTML(html))


def handle_send(_=None):
    question = text_in.value.strip()
    if not question:
        return
    text_in.value = ""
    render_message("user", question)
    result = ask(question)
    meta = f"Language: {result['language']}  |  Sources: {', '.join(result['sources'])}"
    render_message("assistant", result["answer"], meta)


def handle_clear(_=None):
    chat_log.clear_output()


send_btn.on_click(handle_send)
clear_btn.on_click(handle_clear)
try:
    text_in.on_submit(handle_send)
except Exception:
    pass

header = widgets.HTML(
    "<h3 style='margin:6px 0;font-family:sans-serif'>Colombia FAQ Assistant</h3>"
)
controls = widgets.HBox([text_in, send_btn, clear_btn])
display(widgets.VBox([header, chat_log, controls]))

## 14. View saved history

In [15]:
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, encoding="utf-8") as f:
        for i, r in enumerate(json.load(f), 1):
            print(f"{i}. [{r['language']}] Q: {r['question']}")
            print(f"   A: {r['answer']}")
            print(f"   Sources: {', '.join(r['sources'])}\n")
else:
    print("No history yet.")

1. [English] Q: What is the capital of Colombia and which countries border it?
   A: The capital of Colombia is Bogotá. Colombia borders Venezuela and Brazil to the east, Peru to the southeast, Ecuador to the southwest, and Panama to the northwest.
   Sources: 01_overview.txt, 02_geography_climate.txt, 03_history.txt, 07_economy.txt

2. [Portuguese] Q: O que leva a bandeja paisa?
   A: A bandeja paisa leva a bandeja, um prato grande tradicional da região de Antioquia, Colômbia. Esta refeição é conhecida por ser muito rica e variada, incluindo vários ingredientes como feijão, arroz, ovo, batatas fritas, linguiça, carne moída, chouriço, abacaxi e torradas de batata.
   Sources: 04_culture_music_literature.txt, 05_food.txt, 06_tourism.txt

3. [Spanish] Q: Que es Cano Cristales y cuando es mejor visitarlo?
   A: Cano Cristales es conocido como el río de cinco colores o la araña que se derritió, ubicado en La Macarena. Es mejor visitarlo entre julio y noviembre, cuando una planta endémica l

## 15. LangChain version (optional)

Same flow built with LangChain and the same free models. Independent from the rest.

In [16]:
!pip -q install -U langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS as LC_FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import pipeline

docs = []
for name in sorted(os.listdir(CORPUS_DIR)):
    with open(os.path.join(CORPUS_DIR, name), encoding="utf-8") as f:
        docs.append(Document(page_content=f.read(), metadata={"source": name}))

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
fragments = splitter.split_documents(docs)

lc_emb = HuggingFaceEmbeddings(model_name=EMB_MODEL)
vstore = LC_FAISS.from_documents(fragments, lc_emb)
retriever = vstore.as_retriever(search_kwargs={"k": TOP_K})

gen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

/tmp/ipykernel_2434/556191323.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as LC_FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
def ask_langchain(question):
    language = detect_language(question)
    passages = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in passages)
    messages = build_prompt(question, context, language)
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = llm.invoke(prompt)
    return output[len(prompt):].strip() if output.startswith(prompt) else output.strip()


print(ask_langchain("Qual e a regiao cafeteira da Colombia?"))

A região cafeteira da Colômbia é formada pelos Andes ocidentais, onde são cultivados os grãos de café que dão origem aos cafés colombianos de alta qualidade exportados mundialmente.
